# Université du Québec à Chicoutimi (UQAC)
## 8INF926 — Atelier en optimisation avancée
### Projet 1 (Hiver 2026) — Modélisation & Programmation dynamique

> Professeure : Sara Séguin  
Date : mars 2026

> Objectif : (i) modéliser des fonctions de production à partir de données réelles (Matlab), puis (ii) résoudre le problème de chargement optimal par **programmation dynamique** (Python).

**Livrables produits par ce notebook (Partie II)** :
- `partie2_validation_100_resultats_python.csv` (détail par ligne)
- `partie2_validation_100_summary_python.csv` (métriques globales)

## Table des matières (guide de lecture)

1. Partie I — Modélisation (Matlab)
   - Données et prétraitement (calcul de $H_{n,i}$)
   - Modélisation de l’élévation avale $Elav(Q_{tot})$
   - Modélisation des puissances unitaires $P_i(Q_i, H_{n,i})$
2. Partie II — Programmation dynamique (Python)
   - Modèle mathématique (états, décisions, récurrence)
   - Implémentation orientée objet
   - Validation sur les 100 premières lignes + temps de résolution
3. Export / vérifications finales

# Partie I — Modélisation des fonctions de production (Matlab)

Cette partie vise à obtenir des **représentations analytiques** à partir des données du fichier Excel (2013–2017, pas de 2 minutes).
Ces fonctions servent ensuite de **fonction objectif** dans la Partie II (programmation dynamique).

## I.1 Données, variables et prétraitement

**Variables disponibles dans l’Excel** (en-tête):
- `Elav` : élévation avale (m)
- `Qtot` : débit total turbiné (m³/s)
- `Qvan` : débit total déversé (m³/s)
- `Niv Amont` : élévation amont (m)
- `Q1..Q5` : débits unitaires par turbine (m³/s)
- `P1..P5` : puissances unitaires (MW)

**Point important (exploitation / contraintes Partie II)** :
- Si une turbine a `Q_i = 0`, elle est **inactive**.  
- Pour la validation, ce qui compte surtout est le **nombre total de turbines actives**, plus que l’identité exacte des turbines actives.

### Calcul de la hauteur de chute nette
La puissance d’une turbine dépend du débit unitaire et de la hauteur de chute nette.
On calcule, pour chaque turbine $i$, la hauteur de chute nette unitaire à partir des capteurs:
$$
H_{n,i} = N_{amont} - Elav(Q_{tot}) - 0.5\times 10^{-5}\, Q_i^2
$$
où le terme $0.5\times 10^{-5}Q_i^2$ modélise les pertes (données par l’énoncé).

### Nettoyage des données
- Suppression des lignes invalides (`NaN`, `Inf`).
- Pour les modèles unitaires $P_i$, on ne conserve que les observations où la turbine est active (typiquement $Q_i>0$ et $P_i>0$).

**Scripts Matlab utilisés** (workspace):
- `partie1_preparation_donnees.m` : import, calcul de $H_{n,i}$, filtrage, exploration (nuages).
- `partie1_fit3d_turbines_auto.m` : ajustement des surfaces $P_i(Q_i,H_{n,i})$ et comparaison de modèles.
- `partie1_validation_modeles.m` : vérifications/figures de validation.

## I.2 Modélisation de l’élévation avale $Elav(Q_{tot})$

### But
La hauteur de chute nette dépend de l’élévation avale. Or l’élévation avale varie avec le débit total (effet hydraulique).
On modélise donc :
$$
Elav = f_{Elav}(Q_{tot})
$$

### Outil / méthode
L’énoncé suggère le *Curve Fitting Toolbox* de Matlab. En pratique, l’essentiel est d’obtenir une approximation polynomiale fiable; l’ajustement revient à estimer les coefficients par moindres carrés (outil Matlab ou équivalent).

### Démarche de sélection du degré (1 → 10)
Nous avons testé des polynômes de degré 1 à 10, en évaluant :
- **Erreur d’ajustement** : $R^2$, RMSE, SSE et analyse qualitative des résidus.
- **Parcimonie** : préférer un modèle simple si le gain est marginal.
- **Stabilité** : éviter un modèle trop « nerveux » (oscillations), surtout près des bords de la plage de données.
- **Utilisabilité** en optimisation : un modèle plus simple est plus facile à intégrer et moins sensible numériquement.

> En pratique, lorsqu’on augmente le degré, l’erreur d’ajustement sur l’échantillon diminue presque toujours, mais le risque de **sur-ajustement** et d’oscillations augmente. L’objectif ici est un bon compromis précision/simplicité.

### Comparaison visuelle (degrés 1 à 10)
Les figures suivantes (produites sous Matlab) montrent l’ajustement obtenu pour chaque degré. Elles servent à :
- vérifier que la courbe suit la tendance moyenne du nuage de points,
- détecter un sous-ajustement (modèle trop rigide),
- détecter un sur-ajustement (courbe trop oscillante / trop sensible).

**Degré 1** — `images-projet1/elav01.png`  
![Évaluation degré 1](images-projet1/elav01.png)

**Degré 2** — `images-projet1/elav02.png`  
![Évaluation degré 2](images-projet1/elav02.png)

**Degré 3** — `images-projet1/elav03.png`  
![Évaluation degré 3](images-projet1/elav03.png)

**Degré 4** — `images-projet1/elav04.png`  
![Évaluation degré 4](images-projet1/elav04.png)

**Degré 5** — `images-projet1/elav05.png`  
![Évaluation degré 5](images-projet1/elav05.png)

**Degré 6** — `images-projet1/elav06.png`  
![Évaluation degré 6](images-projet1/elav06.png)

**Degré 7** — `images-projet1/elav07.png`  
![Évaluation degré 7](images-projet1/elav07.png)

**Degré 8** — `images-projet1/elav08.png`  
![Évaluation degré 8](images-projet1/elav08.png)

**Degré 9** — `images-projet1/elav09.png`  
![Évaluation degré 9](images-projet1/elav09.png)

**Degré 10** — `images-projet1/elav10.png`  
![Évaluation degré 10](images-projet1/elav10.png)

### Lecture « rapport » des 10 essais
Sans surinterpréter, on retient les tendances classiques suivantes (cohérentes avec les figures):
- **Degré 1** : peut être trop simple si la relation est légèrement courbée → risque de sous-ajustement.
- **Degré 2** : introduit une courbure minimale, souvent suffisante pour ce type de relation hydraulique.
- **Degrés 3–4** : peuvent réduire un peu l’erreur mais au prix de coefficients supplémentaires.
- **Degrés 5–10** : on peut obtenir un meilleur ajustement local, mais au risque d’avoir une courbe plus instable (surtout aux extrémités).

### Justification du choix final (degré 2)
Le **degré 2** a été retenu car il représente le meilleur compromis :
1. Il capture correctement la tendance principale $Elav\,(Q_{tot})$ avec une complexité minimale.
2. Les degrés plus élevés apportent un gain d’erreur **marginal** au regard du coût de complexité.
3. Le modèle reste robuste et stable sur la plage observée, ce qui est crucial car il sera utilisé à l’intérieur de la DP (appelé de très nombreuses fois).
4. Interprétation simple : un comportement globalement quadratique est plausible pour une relation débit–élévation avale sur une plage d’exploitation.

### Modèle retenu
Le modèle final est :
$$
Elav(Q_{tot}) = -1.453\times 10^{-6}\,Q_{tot}^2 + 7.022\times 10^{-3}\,Q_{tot} + 99.98
$$
Qualité d’ajustement (ordre de grandeur observé) : $R^2\approx 0.9763$.

## I.3 Modélisation des puissances unitaires $P_i(Q_i, H_{n,i})$

### Forme générale
Pour chaque turbine $i\in\{1,\dots,5\}$, on approxime une surface de production :
$$
P_i = f_i(Q_i, H_{n,i})
$$
avec $H_{n,i}$ calculée comme en I.1.

### Choix du modèle
Plusieurs surfaces polynomiales ont été testées (ex: `poly22`, `poly23`, `poly32`).
Le modèle retenu pour les 5 turbines est **`poly23`**, car il offre un excellent compromis entre :
- précision (RMSE faible, $R^2$ proche de 1),
- stabilité numérique,
- complexité raisonnable.

La forme `poly23` utilisée est :
$$
\begin{aligned}
P_i(Q_i, H_{n,i}) = &\;c_0 + c_1 Q_i + c_2 H_{n,i} + c_3 Q_i^2 + c_4 Q_iH_{n,i} + c_5 H_{n,i}^2 \\
&+ c_6 Q_i^2H_{n,i} + c_7 Q_iH_{n,i}^2 + c_8 H_{n,i}^3
\end{aligned}
$$

### Qualité d’ajustement (résumé)
- Turbine 1 : $R^2=0.99390$, RMSE $\approx 0.208$ MW
- Turbine 2 : $R^2=0.99726$, RMSE $\approx 0.235$ MW
- Turbine 3 : $R^2=0.99258$, RMSE $\approx 0.200$ MW
- Turbine 4 : $R^2=0.99542$, RMSE $\approx 0.202$ MW
- Turbine 5 : $R^2=0.99471$, RMSE $\approx 0.201$ MW

### Figures (modèles retenus)
Les figures suivantes illustrent, pour chaque turbine, le nuage de points et la surface ajustée (Matlab):

**Turbine 1** — `images-projet1/turbine1.png`  
![Surface retenue Turbine 1](images-projet1/turbine1.png)

**Turbine 2** — `images-projet1/turbine2.png`  
![Surface retenue Turbine 2](images-projet1/turbine2.png)

**Turbine 3** — `images-projet1/turbine3.png`  
![Surface retenue Turbine 3](images-projet1/turbine3.png)

**Turbine 4** — `images-projet1/turbine4.png`  
![Surface retenue Turbine 4](images-projet1/turbine4.png)

**Turbine 5** — `images-projet1/turbine5.png`  
![Surface retenue Turbine 5](images-projet1/turbine5.png)

> Remarque : comme demandé, on ne cherche pas à identifier précisément *quelle* turbine est active en historique, mais à reproduire correctement le **nombre total de turbines actives** lors de la validation (Partie II).

# Partie II — Programmation dynamique (Python)

## II.1 Problème d’optimisation (chargement optimal)

Pour une consigne de débit total $Q_{tot}$ et une élévation amont $N_{amont}$, on cherche une répartition des débits unitaires
$\;\mathbf{Q}=(Q_1,\dots,Q_5)$ maximisant la puissance totale produite.

### Fonction objectif
$$
\max_{Q_1,\dots,Q_5} \; \sum_{i=1}^5 P_i(Q_i, H_{n,i})
$$
avec
$$
H_{n,i} = N_{amont} - Elav(Q_{tot}) - 0.5\times 10^{-5}\,Q_i^2
$$

### Contraintes
$$
\sum_{i=1}^5 Q_i = Q_{tot}, \qquad 0 \le Q_i \le \overline{Q}_i
$$
où $\overline{Q}_i=160$ m³/s par défaut, mais peut être :
- **0** si la turbine est indisponible,
- une valeur **inférieure à 160** si une limitation opérationnelle est imposée.

### Discrétisation
Comme demandé, on discrétise le débit à $\Delta Q = 5$ m³/s. On résout donc un problème discret sur une grille de débits.

## II.2 Programmation dynamique (principe)

On traite les turbines une à une.
- **État** : $q$ = débit restant à répartir.
- **Décision** : $u$ = débit alloué à la turbine courante.
- **Transition** : $q' = q-u$.
- **Terminal** : à la fin, le débit restant doit être nul (sinon solution invalide).

Récurrence (notation type Bellman):
$$
V_i(q) = \max_{u \in U_i(q)} \big[\; r_i(u) + V_{i+1}(q-u) \;\big]
$$
avec $r_i(u)=P_i(u,H_{n,i}(u))$ et $U_i(q)=\{u\in\{0,\Delta Q,\dots\}\,|\,u\le q,\;u\le \overline{Q}_i\}$.

## Exécution (Partie II)

1) Vérifie que le fichier Excel est présent : `./DataProjet2026.xlsx.xlsx`
2) Exécute la cellule **Installation** (une seule fois)
3) Exécute ensuite toutes les cellules dans l’ordre (imports → lecture Excel → modèle → DP → validation)
4) Pour tester plus/moins de cas : modifie `n_test` dans la cellule de validation

Paramètres imposés par l’énoncé:
- $Q_{min}=0$ m³/s
- $Q_{max}=160$ m³/s par turbine (sauf restriction)
- $\Delta Q=5$ m³/s

In [10]:
# Installation (exécuter une seule fois)
import sys
!{sys.executable} -m pip install -q pandas openpyxl
print('Dépendances installées (pandas/openpyxl).')

/bin/bash: line 1: /home/johanu/Downloads/cours_uqac/8INF926: No such file or directory
Dépendances installées (pandas/openpyxl).


In [11]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Any, Optional, Sequence
import csv, math, os, time
from statistics import mean
import pandas as pd

print('Imports chargés OK.')

Imports chargés OK.


## II.3 Lecture Excel (pandas + openpyxl)

Lecture robuste du fichier `.xlsx` (en-tête ligne 3, nettoyage des NaN et colonnes vides).

In [12]:
# ===== 1) Lecture Excel via pandas/openpyxl =====
print('Section 1: lecture Excel via pandas/openpyxl.')

def read_excel_table(file_path: str, header_row_1based: int = 3) -> 'pd.DataFrame':
    # pandas: header = index (0-based) de la ligne d’en-tête
    header = header_row_1based - 1
    df = pd.read_excel(file_path, header=header, engine='openpyxl')
    # Nettoyage léger: suppression des lignes entièrement vides
    df = df.dropna(how='all')
    # Nettoyage colonnes: strings, et on vire les colonnes Excel vides (type "Unnamed: 0")
    df.columns = [str(c).replace('\n', ' ').replace('\r', ' ').strip() for c in df.columns]
    df = df[[c for c in df.columns if not str(c).strip().lower().startswith('unnamed:')]]
    return df

print('Fonction prête: read_excel_table(file_path, header_row_1based=3).')

Section 1: lecture Excel via pandas/openpyxl.
Fonction prête: read_excel_table(file_path, header_row_1based=3).


## II.4 Modèle analytique (réutilisation Partie I)

On ré-encode en Python les fonctions retenues en Partie I:
- $Elav(Q_{tot})$ (polynôme degré 2)
- $P_i(Q_i,H_{n,i})$ via `poly23` (5 turbines)

Ce bloc donne une API simple au solveur DP: `power_unit(...)` et `power_dispatch(...)`.

In [13]:
# ===== 2) Modèle analytique (Partie I) =====
print('Section 2: définition du modèle HydroPlantModel (Elav, Hn, P_i).')
@dataclass
class HydroPlantModel:
    elav_coeff: Tuple[float, float, float] = (-1.453e-6, 7.022e-3, 99.98)
    turb_coeff: List[Tuple[float, ...]] = field(default_factory=lambda: [
        (876.5404, -12.3376, -23.7066, 0.0181, 0.5931, -0.5985, -0.0006, -0.0059, 0.0147),
        (568.5323, -6.5275, -25.9285, 0.0123, 0.3086, 0.1932, -0.0004, -0.0025, 0.0009),
        (-2502.4, 9.9767, 178.7178, -0.0138, -0.4584, -4.2689, 3.6178e-4, 0.0054, 0.0340),
        (3347.6, -18.2111, -224.2315, 0.0234, 0.9174, 4.7799, -7.8865e-4, -0.0101, -0.0334),
        (-1452.7, 6.8356, 94.1139, -0.0131, -0.2594, -2.0887, 3.3149e-4, 0.0024, 0.0162),
    ])

    @staticmethod
    def _loss(qi: float) -> float:
        return 0.5e-5 * qi * qi

    def elav(self, qtot: float) -> float:
        a2, a1, a0 = self.elav_coeff
        return a2 * qtot * qtot + a1 * qtot + a0

    def hnet(self, niv_amont: float, qtot: float, qi: float) -> float:
        return niv_amont - self.elav(qtot) - self._loss(qi)

    def power_unit(self, turbine_idx_1based: int, qi: float, niv_amont: float, qtot: float) -> float:
        if qi <= 0:
            return 0.0
        c = self.turb_coeff[turbine_idx_1based - 1]
        h = self.hnet(niv_amont, qtot, qi)
        p = (
            c[0] + c[1]*qi + c[2]*h + c[3]*qi*qi + c[4]*qi*h + c[5]*h*h +
            c[6]*qi*qi*h + c[7]*qi*h*h + c[8]*h*h*h
        )
        return max(0.0, p)

    def power_dispatch(self, q_vec: Sequence[float], niv_amont: float, qtot: float) -> Tuple[List[float], float]:
        p_vec = [self.power_unit(i + 1, q_vec[i], niv_amont, qtot) for i in range(len(q_vec))]
        return p_vec, float(sum(p_vec))

print('HydroPlantModel prêt.')

Section 2: définition du modèle HydroPlantModel (Elav, Hn, P_i).
HydroPlantModel prêt.


## II.5 Optimisation DP (implémentation)

La classe `DPHydroOptimizer` implémente:
- la table de valeur `V[i][q]` (backward)
- la politique `policy[i][q]`
- la reconstruction (backtracking) pour obtenir $Q_1..Q_5$.

On peut imposer des restrictions en fournissant `qmax_by_turbine` (ex: turbine indisponible → `0`).

In [14]:
# ===== 3) Optimiseur DP =====
print('Section 3: définition de l’optimiseur DP (discrétisation dq).')
@dataclass
class DPHydroOptimizer:
    model: HydroPlantModel
    dq: int = 5
    qmin: int = 0
    qmax_default: int = 160
    n_turbines: int = 5

    def optimize(self, qtot: float, niv_amont: float, qmax_by_turbine: Optional[List[float]] = None) -> Dict:
        if qmax_by_turbine is None:
            qmax_by_turbine = [self.qmax_default] * self.n_turbines

        qtot_d = int(round(qtot / self.dq) * self.dq)
        qmax_d = [int(math.floor(qm / self.dq) * self.dq) for qm in qmax_by_turbine]

        states = list(range(0, qtot_d + self.dq, self.dq))
        n_states = len(states)

        v = [[float('-inf')] * n_states for _ in range(self.n_turbines + 1)]
        policy = [[0] * n_states for _ in range(self.n_turbines)]

        v[self.n_turbines][0] = 0.0

        for i in range(self.n_turbines - 1, -1, -1):
            for s, q_remain in enumerate(states):
                umax = min(q_remain, qmax_d[i])
                best_val, best_u = float('-inf'), 0
                for u in range(self.qmin, umax + self.dq, self.dq):
                    q_next = q_remain - u
                    s_next = q_next // self.dq
                    reward = self.model.power_unit(i + 1, float(u), float(niv_amont), float(qtot_d))
                    cand = reward + v[i + 1][s_next]
                    if cand > best_val:
                        best_val, best_u = cand, u
                v[i][s] = best_val
                policy[i][s] = best_u

        q_opt = [0.0] * self.n_turbines
        q_remain = qtot_d
        for i in range(self.n_turbines):
            s = q_remain // self.dq
            u = policy[i][s]
            q_opt[i] = float(u)
            q_remain -= u

        p_vec, p_tot = self.model.power_dispatch(q_opt, float(niv_amont), float(qtot_d))
        return {
            'qtot_input': float(qtot),
            'qtot_discretized': float(qtot_d),
            'niv_amont': float(niv_amont),
            'qmax_by_turbine': qmax_by_turbine,
            'q_opt': q_opt,
            'p_opt_by_turbine': p_vec,
            'p_opt_total': p_tot,
            'n_active': int(sum(1 for q in q_opt if q > 0)),
            'value0': v[0][qtot_d // self.dq],
        }

print('DPHydroOptimizer prêt. (Paramètre clé: dq = 5 m³/s).')

Section 3: définition de l’optimiseur DP (discrétisation dq).
DPHydroOptimizer prêt. (Paramètre clé: dq = 5 m³/s).


## II.6 Validation sur les 100 premières lignes

Méthode (conforme à l’énoncé):
- On prend les 100 premières lignes (débit total + niveau amont).
- Si une turbine est inactive dans l’Excel ($Q_i=0$), elle est rendue indisponible dans l’optimiseur ($\overline{Q}_i=0$).
- On compare la puissance totale optimisée (DP) à la puissance totale mesurée (Excel).
- On mesure aussi le **temps de résolution** par ligne.

Deux fichiers CSV sont générés: détail + synthèse.

In [15]:
# ===== 4) Validation sur 100 lignes =====
XLSX_PATH = './DataProjet2026.xlsx.xlsx'

print('Section 4: validation DP sur 100 lignes.')
print('Fichier Excel:', XLSX_PATH)
if not os.path.exists(XLSX_PATH):
    raise FileNotFoundError(
        f"Impossible de trouver {XLSX_PATH}. "
        "Place le fichier dans le dossier du notebook, ou modifie XLSX_PATH."
    )

def normalize_name(name: str) -> str:
    return ' '.join(str(name).replace('\n', ' ').replace('\r', ' ').split()).lower()

def pick_column(columns: List[str], aliases: List[str]) -> str:
    norm_map = {normalize_name(c): c for c in columns}
    for a in aliases:
        key = normalize_name(a)
        if key in norm_map:
            return norm_map[key]
    raise KeyError(f'Colonne introuvable. Aliases testés: {aliases}')

def to_float(v) -> float:
    # pandas peut donner des NaN (numpy.float) ou pd.NA
    if v is None or v == '' or (hasattr(pd, 'isna') and pd.isna(v)):
        return float('nan')
    try:
        return float(v)
    except Exception:
        return float('nan')

df = read_excel_table(XLSX_PATH, header_row_1based=3)
print('Nb lignes lues:', len(df))
cols = list(df.columns)
print('Colonnes détectées:')
for c in cols:
    print(' -', c)

rows = df.to_dict(orient='records')

c_qtot = pick_column(cols, ['Qtot (m3/s)', 'Qtot'])
c_niv = pick_column(cols, ['Niv Amont (m)', 'NivAmont (m)', 'Niv Amont'])
c_q = [
    pick_column(cols, ['Q1 (m3/s)', 'Q1']),
    pick_column(cols, ['Q2 (m3/s)', 'Q2']),
    pick_column(cols, ['Q3 (m3/s)', 'Q3']),
    pick_column(cols, ['Q4 (m3/s)', 'Q4']),
    pick_column(cols, ['Q5 (m3/s)', 'Q5']),
 ]
c_p = [
    pick_column(cols, ['P1 (MW)', 'P1']),
    pick_column(cols, ['P2 (MW)', 'P2']),
    pick_column(cols, ['P3 (MW)', 'P3']),
    pick_column(cols, ['P4 (MW)', 'P4']),
    pick_column(cols, ['P5 (MW)', 'P5']),
 ]

model = HydroPlantModel()
optimizer = DPHydroOptimizer(model=model, dq=5)
n_test = min(100, len(rows))

print(f'Paramètres DP: dq={optimizer.dq} m³/s, Qmax défaut={optimizer.qmax_default} m³/s')
print(f'On traite jusqu’à {n_test} lignes (en sautant celles qui ont des NaN).')

out_rows = []
err_mw, err_pct, match_active, solve_times = [], [], [], []

for i in range(n_test):
    r = rows[i]
    qtot = to_float(r[c_qtot])
    niv = to_float(r[c_niv])
    q_real = [to_float(r[x]) for x in c_q]
    p_real_vec = [to_float(r[x]) for x in c_p]

    if any(math.isnan(x) for x in [qtot, niv] + q_real + p_real_vec):
        continue

    # Indisponibilité: si Qi réel = 0, on force Qmax = 0 pour la turbine i
    qmax = [160.0] * 5
    for t in range(5):
        if q_real[t] == 0:
            qmax[t] = 0.0

    t0 = time.perf_counter()
    sol = optimizer.optimize(qtot=qtot, niv_amont=niv, qmax_by_turbine=qmax)
    dt = time.perf_counter() - t0

    p_real = sum(p_real_vec)
    p_opt = sol['p_opt_total']
    e = p_opt - p_real
    ep = (100.0 * e / p_real) if p_real != 0 else float('nan')

    n_act_real = sum(1 for q in q_real if q > 0)
    n_act_opt = sol['n_active']
    is_match = 1 if n_act_real == n_act_opt else 0

    err_mw.append(e)
    if not math.isnan(ep):
        err_pct.append(abs(ep))
    match_active.append(is_match)
    solve_times.append(dt)

    out_rows.append({
        'Row': i + 1,
        'Qtot_input': qtot,
        'Qtot_discretized': sol['qtot_discretized'],
        'NivAmont': niv,
        'P_real_total': p_real,
        'P_opt_total': p_opt,
        'Err_MW': e,
        'Err_pct': ep,
        'Nactive_real': n_act_real,
        'Nactive_opt': n_act_opt,
        'Nactive_match': is_match,
        'SolveTime_s': dt,
        'Q1_opt': sol['q_opt'][0],
        'Q2_opt': sol['q_opt'][1],
        'Q3_opt': sol['q_opt'][2],
        'Q4_opt': sol['q_opt'][3],
        'Q5_opt': sol['q_opt'][4],
        'P1_opt': sol['p_opt_by_turbine'][0],
        'P2_opt': sol['p_opt_by_turbine'][1],
        'P3_opt': sol['p_opt_by_turbine'][2],
        'P4_opt': sol['p_opt_by_turbine'][3],
        'P5_opt': sol['p_opt_by_turbine'][4],
    })

mae = mean(abs(x) for x in err_mw)
rmse = math.sqrt(mean(x*x for x in err_mw))
mape = mean(err_pct) if err_pct else float('nan')
active_rate = 100.0 * mean(match_active)
t_mean_ms = 1000.0 * mean(solve_times)
t_max_ms = 1000.0 * max(solve_times)

print('--- Metriques globales ---')
print(f'MAE puissance totale  : {mae:.4f} MW')
print(f'RMSE puissance totale : {rmse:.4f} MW')
print(f'MAPE puissance totale : {mape:.4f} %')
print(f'Match nb actives      : {active_rate:.2f} %')
print(f'Temps moyen resolution: {t_mean_ms:.3f} ms')
print(f'Temps max resolution  : {t_max_ms:.3f} ms')

detail_path = 'partie2_validation_100_resultats_python.csv'
with open(detail_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(out_rows[0].keys()))
    w.writeheader()
    w.writerows(out_rows)

summary_path = 'partie2_validation_100_summary_python.csv'
summary = {
    'MAE_MW': mae,
    'RMSE_MW': rmse,
    'MAPE_pct': mape,
    'NactiveMatchRate_pct': active_rate,
    'MeanSolveTime_ms': t_mean_ms,
    'MaxSolveTime_ms': t_max_ms,
    'Ncases': len(out_rows),
}
with open(summary_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(summary.keys()))
    w.writeheader()
    w.writerow(summary)

print('Fichiers générés:')
print(' -', detail_path)
print(' -', summary_path)

Section 4: validation DP sur 100 lignes.
Fichier Excel: ./DataProjet2026.xlsx.xlsx
Nb lignes lues: 43824
Colonnes détectées:
 - Elav (m)
 - Qtot (m3/s)
 - Qturb  (m3/s)
 - Qvan  (m3/s)
 - Niv Amont (m)
 - Q1 (m3/s)
 - P1 (MW)
 - Q2 (m3/s)
 - P2 (MW)
 - Q3 (m3/s)
 - P3 (MW)
 - Q4 (m3/s)
 - P4 (MW)
 - Q5 (m3/s)
 - P5 (MW)
Paramètres DP: dq=5 m³/s, Qmax défaut=160 m³/s
On traite jusqu’à 100 lignes (en sautant celles qui ont des NaN).
--- Metriques globales ---
MAE puissance totale  : 43.0330 MW
RMSE puissance totale : 45.1093 MW
MAPE puissance totale : 23.7925 %
Match nb actives      : 100.00 %
Temps moyen resolution: 8.117 ms
Temps max resolution  : 12.043 ms
Fichiers générés:
 - partie2_validation_100_resultats_python.csv
 - partie2_validation_100_summary_python.csv


## Résultats finaux

Visualisation des fichiers CSV générés (optionnel: charge et affiche un aperçu).

In [16]:
# Aperçu des résultats
print('\n=== RéSUMÉ DES RÉSULTATS ===')
print()

# Lecture et affichage du fichier de synthèse
if os.path.exists(summary_path):
    print('>> Fichier:', summary_path)
    df_summary = pd.read_csv(summary_path)
    for col in df_summary.columns:
        val = df_summary[col].iloc[0]
        print(f'   {col}: {val}')
    print()

# Apercu du fichier de détail (premières lignes)
if os.path.exists(detail_path):
    print('>> Fichier:', detail_path)
    df_detail = pd.read_csv(detail_path)
    print(f'   Total: {len(df_detail)} lignes valides traitées')
    print(f'   Colonnes: {list(df_detail.columns)}')
    print('   Premières lignes:')
    print(df_detail.head(3).to_string(index=False))
    print()



=== RéSUMÉ DES RÉSULTATS ===

>> Fichier: partie2_validation_100_summary_python.csv
   MAE_MW: 43.03296694363819
   RMSE_MW: 45.10930570061491
   MAPE_pct: 23.792451002276927
   NactiveMatchRate_pct: 100.0
   MeanSolveTime_ms: 8.11748968961183
   MaxSolveTime_ms: 12.042953007039614
   Ncases: 100

>> Fichier: partie2_validation_100_resultats_python.csv
   Total: 100 lignes valides traitées
   Colonnes: ['Row', 'Qtot_input', 'Qtot_discretized', 'NivAmont', 'P_real_total', 'P_opt_total', 'Err_MW', 'Err_pct', 'Nactive_real', 'Nactive_opt', 'Nactive_match', 'SolveTime_s', 'Q1_opt', 'Q2_opt', 'Q3_opt', 'Q4_opt', 'Q5_opt', 'P1_opt', 'P2_opt', 'P3_opt', 'P4_opt', 'P5_opt']
   Premières lignes:
 Row  Qtot_input  Qtot_discretized   NivAmont  P_real_total  P_opt_total    Err_MW   Err_pct  Nactive_real  Nactive_opt  Nactive_match  SolveTime_s  Q1_opt  Q2_opt  Q3_opt  Q4_opt  Q5_opt    P1_opt    P2_opt  P3_opt    P4_opt    P5_opt
   1  578.005676             580.0 137.899994    179.173747   235.3